# WikiText-103 Training — Colab

Separate session from the engine notebook. This one is long-running and will get
disconnected, so everything durable goes to Drive.

Runtime -> Change runtime type -> **T4 GPU**.

## 0. GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/wikitext-gpt'
import os
os.makedirs(PROJECT + '/data', exist_ok=True)
os.makedirs(PROJECT + '/checkpoints', exist_ok=True)
print('project dir:', PROJECT)

## 1. Clone and install

In [ ]:
GPT_BRANCH = 'engine-export'   # drop after merging to main

%cd /content
!rm -rf wikitext-gpt
!git clone -q -b {GPT_BRANCH} https://github.com/williamclymire-tamu/wikitext-gpt.git
!pip -q install datasets tokenizers

!test -f wikitext-gpt/train.py && grep -q 'resume' wikitext-gpt/train.py && echo "OK resumable train.py" || echo "WRONG BRANCH"

# point the repo's data/ at Drive so tokenization survives a disconnect
!ln -sfn {PROJECT}/data /content/wikitext-gpt/data
%cd /content/wikitext-gpt
!ls -la data/

## 2. Prepare data — run this ONCE, ever

Downloads WikiText-103, trains a 16,384-entry ByteLevelBPE tokenizer over ~100M
tokens, and writes `train/val/test.pt`. Takes 20-30 minutes. Because `data/` is
symlinked to Drive, it survives the session — the guard below skips it on reruns.

Note: tokens are saved as `int64`, so `train.pt` is roughly 900 MB. A 16k vocab
fits in `uint16`, which would cut that 4x. Not required, just wasteful.

In [ ]:
import os
if os.path.exists('data/train.pt'):
    print('already prepared, skipping')
    !ls -la data/
else:
    !python prepare_data.py

## 3. Train

3 epochs, not 10: at 11.6M parameters, roughly compute-optimal is ~230M training
tokens, which is 2-3 passes over WikiText-103. More mostly buys overfitting.

Checkpoints every 500 steps to Drive, carrying optimizer and AMP scaler state.
Keep this browser tab active — Colab disconnects idle sessions after ~90 minutes.

In [ ]:
!python train.py \
    --out-dir {PROJECT}/checkpoints \
    --epochs 3 \
    --batch-size 32 \
    --save-every 500

## 4. Resume after a disconnect

Re-run cells 0 and 1 first (the runtime is gone), then this. It restores model,
optimizer and scaler state, and fast-forwards to the exact batch it died on — the
shuffle generator is seeded per epoch, so the data order is reproduced rather
than approximated.

In [ ]:
!python train.py \
    --out-dir {PROJECT}/checkpoints \
    --epochs 3 \
    --batch-size 32 \
    --save-every 500 \
    --resume

## 5. Evaluate

Perplexity is **token-level over the 16,384-entry BPE vocab**. Published
WikiText-103 numbers are word-level and are not comparable — a smaller vocabulary
mechanically lowers token-level perplexity. Report it with that qualifier.

In [ ]:
!python evaluate.py --checkpoint {PROJECT}/checkpoints/best.pt --split test
!python evaluate.py --checkpoint {PROJECT}/checkpoints/best.pt --split val

## 6. Export the trained model for the engine

This time with the real tokenizer table, so the engine emits actual text.
Copy `export/` to Drive, then load it in the engine notebook.

In [ ]:
!python export_weights.py \
    --checkpoint {PROJECT}/checkpoints/best.pt \
    --tokenizer-dir data/tokenizer \
    --out {PROJECT}/export
!ls {PROJECT}/export | head
!ls {PROJECT}/export/tokenizer

## 7. Sanity check — generate from the trained model

If this produces English-shaped text, training worked. Then re-run the engine
notebook pointing at `{PROJECT}/export` instead of the random export.

In [ ]:
!python generate.py "The history of" \
    --checkpoint {PROJECT}/checkpoints/best.pt \
    --tokenizer-dir data/tokenizer \
    --max-tokens 100 --temperature 0.8 --top-k 40